### Loading different libraries and files

In [49]:
import numpy as np
import joblib
import pandas as pd
loaded = np.load('artifacts/dataset_splits.npz', allow_pickle=True)

X_train = loaded['X_train']
X_test = loaded['X_test']

y_train = loaded['y_train']
y_test = loaded['y_test']

cols = loaded['columns']

## convering the numpy arrays back to pandas dataframes that the preprocessor know the column names of the dataframes
X_train = pd.DataFrame(X_train, columns=cols)
X_test = pd.DataFrame(X_test, columns=cols)

## Loading the preprocessor
preprocessor=joblib.load(filename=r'artifacts\churn_model_pipeline.pkl')
X_train_processed = preprocessor.fit_transform(X_train)

X_train_processed=pd.DataFrame(X_train_processed).astype('float32')
X_train_processed.shape

X_test_processed = preprocessor.transform(X_test)
X_test_processed=pd.DataFrame(X_test_processed).astype('float32')

## ANN implementation

In [50]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard

import datetime

In [51]:
model = Sequential([
    Dense(256, activation='relu', input_dim=X_train_processed.shape[1]),
    Dense(128, activation='relu'),
    Dense(64, activation='relu'),
    Dense(32, activation='relu'),
    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid')
]
)

model.summary()


f:\ANN Deep Learning project\.venv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_36 (Dense)                │ (None, 256)            │         6,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_37 (Dense)                │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_38 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_39 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_40 (Dense)                │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_41 (Dense)                │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 50,177 (196.00 KB)

 Trainable params: 50,177 (196.00 KB)

 Non-trainable params: 0 (0.00 B)

### Model Compilation

In [52]:
opt = tf.keras.optimizers.Adam(learning_rate=0.01)
loss = tf.keras.losses.BinaryCrossentropy(name='binary_crossentropy')

model.compile(optimizer=opt, loss=loss, metrics=['accuracy'])

In [53]:
## setting up the callbacks for early stopping and tensorboard
log_dir = "logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")

tensorflow_callback = TensorBoard(log_dir=log_dir, histogram_freq=1)
early_stopping = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)

history = model.fit(X_train_processed, y_train, validation_data=(X_test_processed, y_test), epochs=100, callbacks=[tensorflow_callback, early_stopping])



Epoch 1/100
238/238 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - accuracy: 0.8330 - loss: 0.4009 - val_accuracy: 0.8413 - val_loss: 0.3598
Epoch 2/100
238/238 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.8522 - loss: 0.3721 - val_accuracy: 0.8534 - val_loss: 0.3498
Epoch 3/100
238/238 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.8547 - loss: 0.3614 - val_accuracy: 0.8607 - val_loss: 0.3215
Epoch 4/100
238/238 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.8577 - loss: 0.3479 - val_accuracy: 0.8586 - val_loss: 0.3299
Epoch 5/100
238/238 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.8642 - loss: 0.3447 - val_accuracy: 0.8571 - val_loss: 0.3377
Epoch 6/100
238/238 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.8625 - loss: 0.3429 - val_accuracy: 0.8518 - val_loss: 0.3321
Epoch 7/100
238/238 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.8631 - loss: 0.3401 - val_accuracy: 0.8523 - val_loss: 0.3446
Epoch 8/100
238/238 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.8630 - loss: 0.3399 - val_accu

### Saving the model

In [54]:
model.save('artifacts/Ann_model.h5')

### Loading Tensorflow Extension

In [55]:
%load_ext tensorboard
%tensorboard --logdir logs/fit

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


Reusing TensorBoard on port 6006 (pid 2848), started 0:01:04 ago. (Use '!kill 2848' to kill it.)